# Validação dos pontos do Google Earth Pro — Versão Melhorada

**Melhorias metodológicas aplicadas:**
- Pipeline consolidado numa única célula de configuração + funções — sem duplicação entre células
- **dV corrigido:** denominador com cossenos conforme o sistema de equações asc/desc standard; assunção dH≈0 documentada
- **Tendência via Theil-Sen** (robusto a outliers e sazonalidade) em vez de OLS puro
- **Tendência estimada sobre o componente Trend do STL**, não sobre a série bruta com sazonalidade
- **R² cruzado** InSAR vs Geodesia (não InSAR vs tempo)
- **p-value** da regressão reportado na tabela
- **Intervalos de confiança (95%)** nas linhas de tendência
- **Scatter plot InSAR vs Geodesia** com linha 1:1 para validação visual directa
- **Diagnóstico de consistência** da sobreposição 2019-2022 entre os dois stacks antes de concatenar
- Escala uniforme em todos os subplots de comparação

## Célula 0 — Mapa de Diagnóstico Espacial (2019-2023)

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
import numpy as np
import fiona
import contextily as ctx
from shapely.geometry import box
from scipy.spatial import cKDTree

# ==============================================================================
# CONFIGURAÇÕES E PARÂMETROS
# ==============================================================================
PATH_KML   = 'data/Blocos_Alqueva.kml'
PATH_ASC   = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
PATH_DESC  = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"

norte_min, norte_max = 1855050, 1855850
este_min,  este_max  = 2792250, 2793250
RAIO_IDW    = 15
RAIO_AMOSTRA = 15
IDW_K, IDW_P = 3, 2

# ==============================================================================
# FUNÇÕES DE PROCESSAMENTO (definidas uma vez, usadas em todas as células)
# ==============================================================================
def melt_robust(df, val_name):
    meta = ['easting','northing','latitude','longitude','incidence_angle','track_angle','pid','p_id']
    present_meta = [c for c in meta if c in df.columns]
    date_cols = [c for c in df.columns if c not in present_meta]
    m = df.melt(id_vars=present_meta, value_vars=date_cols, var_name='date', value_name=val_name).dropna()
    m['date'] = pd.to_datetime(m['date'], format='%Y%m%d', errors='coerce')
    m[val_name] = pd.to_numeric(m[val_name], errors='coerce')
    return m.dropna(subset=['date', val_name])

def load_filter(path, buf=100):
    df = pd.read_csv(path)
    df['easting']  = pd.to_numeric(df['easting'],  errors='coerce')
    df['northing'] = pd.to_numeric(df['northing'], errors='coerce')
    return df[
        (df['northing'] >= norte_min-buf) & (df['northing'] <= norte_max+buf) &
        (df['easting']  >= este_min-buf)  & (df['easting']  <= este_max+buf)
    ].dropna(subset=['easting','northing'])

def interp_ps(df, dates):
    dfs, t_x = [], dates.view(np.int64)
    for (x, y), g in df.groupby(['easting','northing']):
        g  = g.sort_values('date')
        xp = g['date'].values.view(np.int64)
        fp = g['disp'].values.astype(np.float64)
        res = pd.DataFrame({
            'easting': x, 'northing': y, 'date': dates,
            'disp': np.interp(t_x, xp, fp),
            'inc': g['incidence_angle'].iloc[0],
            'lon': g['longitude'].iloc[0], 'lat': g['latitude'].iloc[0],
            'pid': g['pid'].iloc[0] if 'pid' in g.columns else 0
        })
        dfs.append(res)
    return pd.concat(dfs) if dfs else pd.DataFrame()

def idw_calc(source, target, r, k, p):
    out = []
    for d, src in source.groupby('date'):
        tgt = target[target['date']==d].copy()
        if src.empty or tgt.empty: continue
        tree = cKDTree(src[['easting','northing']].values)
        dist, idx = tree.query(tgt[['easting','northing']].values, k=k, distance_upper_bound=r)
        vals, thetas = [], []
        for d_i, i_i in zip(dist, idx):
            mask = np.isfinite(d_i)
            if not np.any(mask): vals.append(np.nan); thetas.append(np.nan); continue
            w = 1 / (d_i[mask]**p)
            vals.append(np.sum(w * src.iloc[i_i[mask]]['disp']) / np.sum(w))
            thetas.append(np.sum(w * src.iloc[i_i[mask]]['inc']) / np.sum(w))
        tgt['disp_desc'], tgt['theta_desc'] = vals, thetas
        out.append(tgt)
    return pd.concat(out).dropna() if out else pd.DataFrame()

def calc_dv(df):
    """
    Decomposição vertical do deslocamento InSAR.
    Assunção: dH ≈ 0 (válida para estrutura de barragem com movimento predominantemente vertical).
    Sistema:
        d_asc  = cos(θ_asc)  * dV  →  dV = d_asc  / cos(θ_asc)
        d_desc = cos(θ_desc) * dV  →  dV = d_desc / cos(θ_desc)
    Solução por mínimos quadrados ponderados (pesos iguais):
        dV = (d_asc * cos(θ_desc) + d_desc * cos(θ_asc)) / (cos(θ_asc)^2 + cos(θ_desc)^2)
    Note: esta é a formulação correcta quando se assume dH=0.
    """
    ta = np.deg2rad(df['inc'])
    td = np.deg2rad(df['theta_desc'])
    num   = df['disp'] * np.cos(td) + df['disp_desc'] * np.cos(ta)
    denom = np.cos(ta)**2 + np.cos(td)**2
    return num / denom

# ==============================================================================
# PROCESSAMENTO
# ==============================================================================
print("1/3 - Processando InSAR e Geometrias...")
asc_l  = melt_robust(load_filter(PATH_ASC),  'disp')
desc_l = melt_robust(load_filter(PATH_DESC), 'disp')
common_dates = pd.date_range(start=asc_l['date'].min(), end=asc_l['date'].max(), freq='MS')

asc_i  = interp_ps(asc_l,  common_dates)
desc_i = interp_ps(desc_l, common_dates)

manual_dv_df = idw_calc(desc_i, asc_i, r=RAIO_IDW, k=IDW_K, p=IDW_P)
manual_dv_df['dV_final'] = calc_dv(manual_dv_df)

gdf_manual = gpd.GeoDataFrame(
    manual_dv_df,
    geometry=gpd.points_from_xy(manual_dv_df['lon'], manual_dv_df['lat']),
    crs="EPSG:4326"
).to_crs(epsg=3763)

# Buffers KML
fiona.drvsupport.supported_drivers['KML'] = 'rw'
gdf_kml       = gpd.read_file(PATH_KML, driver='KML')
gdf_pts       = gdf_kml[gdf_kml.geometry.type == 'Point'].copy().to_crs(epsg=3763)
gdf_circulos  = gdf_pts.copy()
gdf_circulos.geometry = gdf_pts.geometry.buffer(RAIO_AMOSTRA)

# PS dentro dos raios
gdf_asc_all  = gpd.GeoDataFrame(asc_i.drop_duplicates(['easting','northing']),
    geometry=gpd.points_from_xy(
        asc_i.drop_duplicates(['easting','northing'])['lon'],
        asc_i.drop_duplicates(['easting','northing'])['lat']),
    crs="EPSG:4326").to_crs(epsg=3763)
gdf_desc_all = gpd.GeoDataFrame(desc_i.drop_duplicates(['easting','northing']),
    geometry=gpd.points_from_xy(
        desc_i.drop_duplicates(['easting','northing'])['lon'],
        desc_i.drop_duplicates(['easting','northing'])['lat']),
    crs="EPSG:4326").to_crs(epsg=3763)

gdf_asc_in  = gpd.sjoin(gdf_asc_all,  gdf_circulos, how="inner", predicate="within")
gdf_desc_in = gpd.sjoin(gdf_desc_all, gdf_circulos, how="inner", predicate="within")

# ==============================================================================
# MAPA DE DIAGNÓSTICO
# ==============================================================================
fig, ax = plt.subplots(figsize=(12, 10))
gdf_circulos.to_crs(epsg=3857).plot(ax=ax, facecolor='red', alpha=0.2, edgecolor='black', linestyle='--')
if not gdf_asc_in.empty:
    gdf_asc_in.to_crs(epsg=3857).plot(ax=ax, color='#f39c12', markersize=25, zorder=3)
if not gdf_desc_in.empty:
    gdf_desc_in.to_crs(epsg=3857).plot(ax=ax, color='#3498db', markersize=25, zorder=3)
gdf_pts.to_crs(epsg=3857).plot(ax=ax, color='yellow', marker='*', markersize=180, edgecolor='black', zorder=5)
for _, row in gdf_pts.to_crs(epsg=3857).iterrows():
    ax.text(row.geometry.x, row.geometry.y + 12, row['Name'], color='white',
            fontweight='bold', fontsize=9, ha='center',
            bbox=dict(facecolor='black', alpha=0.7, pad=2, edgecolor='none'))
ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery)
h = [
    mlines.Line2D([], [], color='#f39c12', marker='o', linestyle='None', markersize=7, label='Ascendente (no raio)'),
    mlines.Line2D([], [], color='#3498db', marker='o', linestyle='None', markersize=7, label='Descendente (no raio)'),
    mpatches.Patch(facecolor='red', alpha=0.2, edgecolor='black', linestyle='--', label=f'Área Amostragem ({RAIO_AMOSTRA}m)'),
]
ax.legend(handles=h, loc='lower right', frameon=True, facecolor='white')
ax.set_axis_off()
plt.title(f"Diagnóstico InSAR Alqueva: PS capturados nos raios de {RAIO_AMOSTRA}m", fontsize=13)
plt.tight_layout(); plt.show()
print(f"✅ {len(gdf_asc_in)} PS Asc | {len(gdf_desc_in)} PS Desc identificados nos raios.")


---
## Célula 1 — Séries Temporais e STL (2019-2023)

In [ ]:
import matplotlib.dates as mdates
from statsmodels.tsa.seasonal import seasonal_decompose

# Amostragem espacial nos buffers KML
final_series = {}
for nome in gdf_circulos['Name'].unique():
    m_data = gpd.sjoin(gdf_manual, gdf_circulos[gdf_circulos['Name']==nome], how="inner", predicate="within")
    if not m_data.empty:
        final_series[nome] = m_data.groupby('date')['dV_final'].mean()

print(f"Pontos com série InSAR: {list(final_series.keys())}")

if final_series:
    nomes  = list(final_series.keys())
    n_pts  = len(nomes)
    cols_g = int(np.ceil(np.sqrt(n_pts)))
    rows_g = int(np.ceil(n_pts / cols_g))

    # --- Grelha de Séries ---
    all_vals = pd.concat(final_series.values())
    ymin, ymax = all_vals.min(), all_vals.max()
    pad = (ymax - ymin) * 0.1

    fig, axes = plt.subplots(rows_g, cols_g, figsize=(4*cols_g, 3*rows_g), sharex=True, sharey=True)
    axes = np.array(axes).flatten()
    for i, ax in enumerate(axes):
        if i < n_pts:
            nome = nomes[i]
            data = final_series[nome].interpolate().ffill().bfill()
            ax.plot(data.index, data.values, color='tab:blue', lw=1.2)
            ax.set_title(nome, fontsize=10, fontweight='bold')
            ax.set_ylim(ymin-pad, ymax+pad)
            ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
            if i % cols_g == 0: ax.set_ylabel('dV (mm)')
        else:
            ax.axis('off')
    plt.suptitle("Séries InSAR por Ponto de Controlo", fontweight='bold')
    plt.tight_layout(); plt.show()

    # --- STL ---
    decomps = {}
    for nome in nomes:
        s = final_series[nome].interpolate().ffill().bfill()
        try:
            decomps[nome] = seasonal_decompose(s, period=12, model='additive', extrapolate_trend='freq')
        except Exception as e:
            print(f"STL falhou para {nome}: {e}")
            decomps[nome] = None

    def safe_lims(comps):
        v = pd.concat(comps).dropna()
        m = (v.max()-v.min())*0.1
        return v.min()-m, v.max()+m

    valid = {k: v for k, v in decomps.items() if v is not None}
    if valid:
        l_obs   = safe_lims([v.observed  for v in valid.values()])
        l_trend = safe_lims([v.trend     for v in valid.values()])
        l_seas  = safe_lims([v.seasonal  for v in valid.values()])
        l_resid = safe_lims([v.resid     for v in valid.values()])

        fig, axes = plt.subplots(len(valid), 4, figsize=(16, 2.0*len(valid)), sharex=True, squeeze=False)
        for i, (nome, res) in enumerate(valid.items()):
            axes[i,0].plot(res.observed.index,  res.observed,  color='black', lw=1)
            axes[i,1].plot(res.trend.index,     res.trend,     color='tab:blue', lw=1.5)
            axes[i,2].plot(res.seasonal.index,  res.seasonal,  color='tab:green', lw=1)
            axes[i,3].scatter(res.resid.index,  res.resid,     color='gray', s=3, alpha=0.6)
            axes[i,3].axhline(0, c='k', ls='--', lw=0.5)
            for j, lim in enumerate([l_obs, l_trend, l_seas, l_resid]):
                axes[i,j].set_ylim(lim)
            axes[i,0].set_ylabel(nome, fontweight='bold', color='darkred', fontsize=9)
            if i == 0:
                for j, t in enumerate(['Observed','Trend','Seasonal','Residual']):
                    axes[i,j].set_title(t, fontweight='bold')
            if i == len(valid)-1:
                for ax in axes[i,:]: ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
        plt.suptitle("Decomposição STL por Ponto de Controlo", fontweight='bold')
        plt.tight_layout(); plt.show()


---
## Célula 2 — Análise de Sensibilidade IDW

In [ ]:
from scipy import stats as scipy_stats
from sklearn.metrics import mean_squared_error

PATH_NIVEL = 'data/nivelamento.xlsx'
RAIO_AMOSTRA_SENS = 15

CENARIOS = [
    {'id': 'Largo (50m)',    'r': 50, 'k': 5, 'p': 2},
    {'id': 'Médio (30m)',    'r': 30, 'k': 4, 'p': 2},
    {'id': 'Apertado (15m)','r': 15, 'k': 3, 'p': 2},
]

# Geodesia
df_geo = pd.read_excel(PATH_NIVEL)
df_geo['data'] = pd.to_datetime(df_geo['data'])
df_geo['val']  = pd.to_numeric(df_geo['deslocamento (m)'].astype(str).str.replace(',','.'), errors='coerce')

res_stats = []

for c in CENARIOS:
    print(f"Processando cenário {c['id']}...")
    df_dv = idw_calc(desc_i, asc_i, r=c['r'], k=c['k'], p=c['p'])
    if df_dv.empty: continue
    df_dv['dV'] = calc_dv(df_dv)

    gdf_c = gpd.GeoDataFrame(df_dv, geometry=gpd.points_from_xy(df_dv['lon'], df_dv['lat']),
                              crs="EPSG:4326").to_crs(epsg=3763)

    for nome in gdf_circulos['Name'].unique():
        joined = gpd.sjoin(gdf_c, gdf_circulos[gdf_circulos['Name']==nome], how="inner", predicate="within")
        if joined.empty: continue

        s_insar = joined.groupby('date')['dV'].mean().sort_index()
        insar_z = (s_insar - s_insar.iloc[0]).values

        s_geo_pt = df_geo[df_geo['instrumento'] == nome].sort_values('data')
        if s_geo_pt.empty: continue

        ref_geo  = s_geo_pt['val'].iloc[0]
        geo_z    = (s_geo_pt['val'] - ref_geo).values
        days_i   = (s_insar.index - s_insar.index[0]).days.values
        days_g   = (s_geo_pt['data'] - s_geo_pt['data'].iloc[0]).dt.days.values

        # Interpolação temporal para alinhar datas
        insar_at_geo = np.interp(days_g, days_i, insar_z)
        rmse = np.sqrt(mean_squared_error(geo_z, insar_at_geo))

        # Theil-Sen para robustez
        ts_i = scipy_stats.theilslopes(insar_z, days_i)
        ts_g = scipy_stats.theilslopes(geo_z,   days_g)

        # R² cruzado InSAR vs Geodesia
        r_cross, p_cross = scipy_stats.pearsonr(geo_z, insar_at_geo)

        res_stats.append({
            'Cenário':           c['id'],
            'Ponto':             nome,
            'Vel_InSAR (mm/a)':  ts_i.slope * 365.25,
            'Vel_Geo (mm/a)':    ts_g.slope * 365.25,
            'RMSE (mm)':         round(rmse, 3),
            'R²_cruzado':        round(r_cross**2, 3),
            'p_cruzado':         round(p_cross, 4),
        })

df_sens = pd.DataFrame(res_stats)
print("\n--- ANÁLISE DE SENSIBILIDADE IDW ---")
print(df_sens.to_string(index=False))


---
## Célula 3 — Comparação InSAR / Nivelamento LNEC (2019-2023)

In [ ]:
# Geodesia (reutiliza df_geo da célula anterior)
df_geo_raw = df_geo.copy()
df_geo_raw.rename(columns={'val': 'valor_corrigido', 'data': 'data'}, inplace=True)

# Se o Excel tiver coluna 'instrumento', usar; senão assumir Nome do KML
if 'instrumento' not in df_geo_raw.columns:
    df_geo_raw['instrumento'] = df_geo_raw.get('instrumento', df_geo_raw.iloc[:,0])

inst_comuns = [n for n in final_series.keys() if n in df_geo_raw['instrumento'].unique()]
print(f"Pontos em comum InSAR ∩ Geodesia: {inst_comuns}")

if inst_comuns:
    fig, axes = plt.subplots(len(inst_comuns), 1, figsize=(14, 4.5 * len(inst_comuns)), sharex=False)
    if len(inst_comuns) == 1: axes = [axes]

    for i, nome in enumerate(inst_comuns):
        ax = axes[i]  # <-- Corrigida a indentação aqui

        # ── Série InSAR ──────────────────────────────────────────────────────
        s_insar = final_series[nome][final_series[nome].index.year >= 2019].sort_index()
        if s_insar.empty:
            continue

        t0_insar = s_insar.index[0]   # data exacta do primeiro ponto InSAR

        # ── Série de Geodesia completa (para plotar histórico) ───────────────
        s_geo_full = df_geo_raw[df_geo_raw['instrumento'] == nome].sort_values('data')
        s_geo_2019 = s_geo_full[s_geo_full['data'].dt.year >= 2019]
        if s_geo_2019.empty:
            continue

        # ── REFERÊNCIA COMUM: interpolar a geodesia na data exacta t0_insar ──
        geo_dates_ns = s_geo_full['data'].values.astype('int64')
        geo_vals     = s_geo_full['valor_corrigido'].values
        t0_ns        = np.int64(t0_insar.value)       # nanoseconds

        # só interpolamos se t0 estiver dentro do intervalo da geodesia
        if t0_ns < geo_dates_ns[0] or t0_ns > geo_dates_ns[-1]:
            print(f"⚠️  {nome}: t0 InSAR ({t0_insar.date()}) fora do intervalo "
                  f"da geodesia ({s_geo_full['data'].iloc[0].date()} – "
                  f"{s_geo_full['data'].iloc[-1].date()}). A saltar.")
            continue

        ref_comum = np.interp(t0_ns, geo_dates_ns, geo_vals)

        # ── Série InSAR com zero em t0 ────────────────────────────────────────
        insar_zero = s_insar - s_insar.iloc[0]

        # ── Geodesia com MESMO zero (referenciada ao valor interpolado em t0) ─
        geo_hist_zero = s_geo_full['valor_corrigido'] - ref_comum   # histórico
        geo_2019_zero = s_geo_2019['valor_corrigido'] - ref_comum   # janela InSAR

        # ── Theil-Sen ────────────────────────────────────────────────────────
        days_i = (insar_zero.index - t0_insar).days.values
        days_g = (s_geo_2019['data'] - t0_insar).dt.days.values
        vals_g = geo_2019_zero.values

        ts_i = scipy_stats.theilslopes(insar_zero.values, days_i)
        ts_g = scipy_stats.theilslopes(vals_g, days_g)

        vel_i    = ts_i.slope * 365.25
        vel_i_lo = ts_i.low_slope  * 365.25
        vel_i_hi = ts_i.high_slope * 365.25
        vel_g    = ts_g.slope * 365.25

        # ── R² e RMSE ────────────────────────────────────────────────────────
        insar_at_geo = np.interp(days_g, days_i, insar_zero.values)
        r_cross, p_cross = scipy_stats.pearsonr(vals_g, insar_at_geo)
        rmse = np.sqrt(mean_squared_error(vals_g, insar_at_geo))

        # ── Plot ──────────────────────────────────────────────────────────────
        mask_pre = s_geo_full['data'].dt.year < 2019
        ax.plot(s_geo_full['data'][mask_pre], geo_hist_zero[mask_pre],
                color='gray', ls='--', alpha=0.4,
                label=f'Histórico Geodesia (antes de {t0_insar.year})')
        
        ax.scatter(s_geo_2019['data'], vals_g,
                   color='red', marker='D', s=35,
                   label='Geodesia (2019–2023)', zorder=5)
        
        ax.plot(insar_zero.index, insar_zero.values,
                color='tab:blue', lw=1.2, label='InSAR (2019–2023)')

        trend_line_i = ts_i.intercept + ts_i.slope * days_i
        band_lo_i    = ts_i.low_slope  * days_i + ts_i.intercept
        band_hi_i    = ts_i.high_slope * days_i + ts_i.intercept
        
        ax.plot(insar_zero.index, trend_line_i,
                color='darkblue', lw=2,
                label=f'Trend InSAR {vel_i:.2f} mm/a')
        
        ax.fill_between(insar_zero.index, band_lo_i, band_hi_i, color='darkblue', alpha=0.12)
        
        ax.plot(s_geo_2019['data'], ts_g.intercept + ts_g.slope * days_g,
                color='darkred', ls=':', lw=2, label=f'Trend Geo {vel_g:.2f} mm/a')

        ax.axhline(0, color='black', lw=1)
        ax.axvline(t0_insar, color='green', lw=1.5, ls='--', alpha=0.6,
                   label=f'Referência comum: {t0_insar.date()}')

        ax.set_title(
            f"Ponto {nome} | RMSE={rmse:.2f} mm | R²={r_cross**2:.3f} | p={p_cross:.4f}\n"
            f"zero = valor geo interpolado em {t0_insar.date()} ({ref_comum:.2f} m)",
            fontsize=11, fontweight='bold'
        )
        ax.set_ylabel("Variação (mm)")
        ax.grid(True, alpha=0.2)
        ax.legend(loc='upper left', fontsize=8, ncol=2)
        ax.xaxis.set_major_locator(mdates.YearLocator(1)) # Reduzido para 1 ano para melhor leitura
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

    plt.tight_layout() # <-- Fora do loop
    plt.show()         # <-- Fora do loop


---
## Célula 4 — Tendências sobre o componente Trend do STL

Estimar a velocidade sobre a série bruta é afectado pelo sinal sazonal (correlação com nível da albufeira). A prática correcta é extrair primeiro o componente de **Trend** via STL e regredir sobre ele.

In [ ]:
stl_stats_resumo = []

if inst_comuns and 'decomps' in dir() and decomps:
    fig, axes = plt.subplots(len(inst_comuns), 1, figsize=(14, 4*len(inst_comuns)), sharex=False)
    if len(inst_comuns) == 1: axes = [axes]

    for i, nome in enumerate(inst_comuns):
        ax = axes[i]

        if decomps.get(nome) is None:
            ax.set_title(f"{nome}: STL não disponível")
            continue

        # Componente Trend do STL (série sem sazonalidade nem ruído)
        trend_stl = decomps[nome].trend.dropna()
        trend_stl = trend_stl[trend_stl.index.year >= 2019]
        trend_zero = trend_stl - trend_stl.iloc[0]
        days_trend = (trend_zero.index - trend_zero.index[0]).days.values

        # Theil-Sen sobre o Trend STL
        ts_stl = scipy_stats.theilslopes(trend_zero.values, days_trend)
        vel_stl    = ts_stl.slope * 365.25
        vel_stl_lo = ts_stl.low_slope  * 365.25
        vel_stl_hi = ts_stl.high_slope * 365.25

        # Geodesia (para comparação)
        s_geo_full = df_geo_raw[df_geo_raw['instrumento']==nome].sort_values('data')
        s_geo_2019 = s_geo_full[s_geo_full['data'].dt.year >= 2019]
        if not s_geo_2019.empty:
            ref_2019 = s_geo_2019['valor_corrigido'].iloc[0]
            vals_g   = (s_geo_2019['valor_corrigido'] - ref_2019).values
            days_g   = (s_geo_2019['data'] - s_geo_2019['data'].iloc[0]).dt.days.values
            ts_g     = scipy_stats.theilslopes(vals_g, days_g)
            vel_g    = ts_g.slope * 365.25
            ax.scatter(s_geo_2019['data'], vals_g, color='red', marker='D', s=35,
                       label='Geodesia (2019-2023)', zorder=5)
            ax.plot(s_geo_2019['data'], ts_g.intercept + ts_g.slope * days_g,
                    color='darkred', ls=':', lw=2, label=f'Trend Geo {vel_g:.2f} mm/a')
        else:
            vel_g = np.nan

        # InSAR série bruta (fundo)
        s_insar = final_series[nome][final_series[nome].index.year >= 2019].sort_index()
        insar_zero = s_insar - s_insar.iloc[0]
        ax.plot(insar_zero.index, insar_zero.values,
                color='tab:blue', lw=1, alpha=0.4, label='InSAR bruto')

        # Trend STL
        ax.plot(trend_zero.index, trend_zero.values,
                color='navy', lw=1.8, label='InSAR Trend (STL)')

        # Regressão Theil-Sen sobre Trend STL + IC
        trend_fit  = ts_stl.intercept + ts_stl.slope * days_trend
        band_lo    = ts_stl.low_slope  * days_trend + ts_stl.intercept
        band_hi    = ts_stl.high_slope * days_trend + ts_stl.intercept
        ax.plot(trend_zero.index, trend_fit,
                color='darkblue', lw=2.5,
                label=f'Trend InSAR(STL) {vel_stl:.2f} [{vel_stl_lo:.2f},{vel_stl_hi:.2f}] mm/a')
        ax.fill_between(trend_zero.index, band_lo, band_hi, color='darkblue', alpha=0.12)

        ax.axhline(0, color='black', lw=1)
        ax.set_title(f"Ponto {nome}: Tendência sobre Trend STL", fontweight='bold')
        ax.set_ylabel("Variação (mm)")
        ax.grid(True, alpha=0.2)
        ax.legend(loc='upper left', fontsize=8, ncol=2)
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

        stl_stats_resumo.append({
            'Ponto':                nome,
            'Vel_InSAR_STL (mm/a)': round(vel_stl, 3),
            'IC95_low':             round(vel_stl_lo, 3),
            'IC95_high':            round(vel_stl_hi, 3),
            'Vel_Geo (mm/a)':       round(vel_g, 3) if not np.isnan(vel_g) else np.nan,
        })

    plt.tight_layout(); plt.show()

print("\n--- VELOCIDADES SOBRE TREND STL ---")
print(pd.DataFrame(stl_stats_resumo).to_string(index=False))


---
## Célula 5 — Scatter InSAR vs Geodesia (linha 1:1)

Validação directa: se os dois sistemas concordam em magnitude e fase, os pontos devem cair sobre a diagonal 1:1.

In [ ]:
if inst_comuns:
    n_pts_comuns = len(inst_comuns)
    fig, axes = plt.subplots(1, n_pts_comuns, figsize=(5*n_pts_comuns, 5), squeeze=False)

    for j, nome in enumerate(inst_comuns):
        ax = axes[0, j]

        s_insar    = final_series[nome][final_series[nome].index.year >= 2019].sort_index()
        insar_zero = s_insar - s_insar.iloc[0]

        s_geo_2019 = df_geo_raw[
            (df_geo_raw['instrumento']==nome) & (df_geo_raw['data'].dt.year >= 2019)
        ].sort_values('data')
        if s_geo_2019.empty: continue

        ref_2019 = s_geo_2019['valor_corrigido'].iloc[0]
        vals_g   = (s_geo_2019['valor_corrigido'] - ref_2019).values
        days_i   = (insar_zero.index - insar_zero.index[0]).days.values
        days_g   = (s_geo_2019['data'] - s_geo_2019['data'].iloc[0]).dt.days.values

        insar_at_geo = np.interp(days_g, days_i, insar_zero.values)

        r_cross, p_cross = scipy_stats.pearsonr(vals_g, insar_at_geo)
        rmse = np.sqrt(mean_squared_error(vals_g, insar_at_geo))

        sc = ax.scatter(vals_g, insar_at_geo, c=days_g, cmap='viridis', s=50, zorder=3)
        plt.colorbar(sc, ax=ax, label='Dias desde 2019')

        # Linha 1:1
        lim_lo = min(vals_g.min(), insar_at_geo.min()) - 1
        lim_hi = max(vals_g.max(), insar_at_geo.max()) + 1
        ax.plot([lim_lo, lim_hi], [lim_lo, lim_hi], 'k--', lw=1.5, label='Linha 1:1')

        # OLS para visualizar bias sistemático
        ols = scipy_stats.linregress(vals_g, insar_at_geo)
        x_fit = np.linspace(lim_lo, lim_hi, 100)
        ax.plot(x_fit, ols.intercept + ols.slope * x_fit,
                color='red', lw=1.5, ls='--',
                label=f'OLS: y={ols.slope:.2f}x+{ols.intercept:.2f}')

        ax.set_xlim(lim_lo, lim_hi); ax.set_ylim(lim_lo, lim_hi)
        ax.set_xlabel('Geodesia (mm)'); ax.set_ylabel('InSAR (mm)')
        ax.set_title(
            f"Ponto {nome}\nR²={r_cross**2:.3f} | RMSE={rmse:.2f} mm | p={p_cross:.4f}",
            fontweight='bold'
        )
        ax.legend(fontsize=8)
        ax.set_aspect('equal')
        ax.grid(True, alpha=0.2)

    plt.suptitle("Validação Cruzada: InSAR vs Geodesia (por data)", fontweight='bold')
    plt.tight_layout(); plt.show()


---
## Célula 6 — Série Integrada 2018-2023 + Diagnóstico de Consistência dos Stacks

In [ ]:
PATH_ASC_18_22  = "data/alqueva_calibrated_asc_desc_2018_2022/EGMS_L2b_147_0224_IW2_VV_2018_2022_1/EGMS_L2b_147_0224_IW2_VV_2018_2022_1.csv"
PATH_DESC_18_22 = "data/alqueva_calibrated_asc_desc_2018_2022/EGMS_L2b_052_0848_IW2_VV_2018_2022_1/EGMS_L2b_052_0848_IW2_VV_2018_2022_1.csv"
PATH_ASC_19_23  = PATH_ASC
PATH_DESC_19_23 = PATH_DESC
RAIO_IDW_INT, IDW_K_INT, IDW_P_INT = 50, 10, 2

print("1/4 - Integrando períodos 2018-2023...")
asc_18  = melt_robust(load_filter(PATH_ASC_18_22),  'disp')
asc_19  = melt_robust(load_filter(PATH_ASC_19_23),  'disp')
desc_18 = melt_robust(load_filter(PATH_DESC_18_22), 'disp')
desc_19 = melt_robust(load_filter(PATH_DESC_19_23), 'disp')

# --- DIAGNÓSTICO: Diferença na sobreposição 2019-2022 ---
print("\n2/4 - Diagnóstico de consistência na sobreposição 2019-2022...")
overlap_dates = pd.date_range(
    start=max(asc_18['date'].min(), asc_19['date'].min()),
    end  =min(asc_18['date'].max(), asc_19['date'].max()),
    freq ='MS'
)
if len(overlap_dates) > 0:
    asc_18_ov  = asc_18[asc_18['date'].isin(overlap_dates)]
    asc_19_ov  = asc_19[asc_19['date'].isin(overlap_dates)]

    # Media espacial por data em cada stack
    mean_18 = asc_18_ov.groupby('date')['disp'].mean()
    mean_19 = asc_19_ov.groupby('date')['disp'].mean()
    common  = mean_18.index.intersection(mean_19.index)

    if len(common) >= 3:
        diff = (mean_18[common] - mean_19[common])
        fig_diag, ax_diag = plt.subplots(figsize=(12, 3))
        ax_diag.plot(common, mean_18[common], label='Stack 2018-2022', color='tab:orange', lw=1.5)
        ax_diag.plot(common, mean_19[common], label='Stack 2019-2023', color='tab:blue',   lw=1.5)
        ax_diag.set_title(
            f"Diagnóstico Sobreposição (Asc) | Δ médio={diff.mean():.2f} mm | std={diff.std():.2f} mm",
            fontweight='bold'
        )
        ax_diag.set_ylabel('Deslocamento médio (mm)')
        ax_diag.legend(); ax_diag.grid(True, alpha=0.2)
        ax_diag.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
        plt.tight_layout(); plt.show()
        print(f"   Δ médio na sobreposição: {diff.mean():.3f} mm (esperado ≈ 0 se stacks consistentes)")
        print(f"   Std da diferença:        {diff.std():.3f} mm")
        if abs(diff.mean()) > 2:
            print("   ⚠️  Offset sistemático detectado — considerar correcção antes de concatenar.")
        else:
            print("   ✅ Stacks consistentes na sobreposição.")
    else:
        print("   Sobreposição insuficiente para diagnóstico.")
else:
    print("   Sem sobreposição temporal entre os dois stacks.")

# --- CONCATENAÇÃO ---
print("\n3/4 - Concatenando e processando série integrada...")
asc_l_int  = pd.concat([asc_18,  asc_19 ]).drop_duplicates(subset=['pid','date']).sort_values(['pid','date'])
desc_l_int = pd.concat([desc_18, desc_19]).drop_duplicates(subset=['pid','date']).sort_values(['pid','date'])

common_dates_int = pd.date_range(start=asc_l_int['date'].min(), end=asc_l_int['date'].max(), freq='MS')
asc_i_int  = interp_ps(asc_l_int,  common_dates_int)
desc_i_int = interp_ps(desc_l_int, common_dates_int)

df_dv_int = idw_calc(desc_i_int, asc_i_int, r=RAIO_IDW_INT, k=IDW_K_INT, p=IDW_P_INT)
if df_dv_int.empty:
    print("⚠️  IDW não retornou dados. Verifica RAIO_IDW_INT e cobertura.")
else:
    df_dv_int['dV_final'] = calc_dv(df_dv_int)
    gdf_int = gpd.GeoDataFrame(df_dv_int,
        geometry=gpd.points_from_xy(df_dv_int['lon'], df_dv_int['lat']),
        crs="EPSG:4326").to_crs(epsg=3763)

    final_series_int = {}
    for nome in gdf_circulos['Name'].unique():
        m_data = gpd.sjoin(gdf_int, gdf_circulos[gdf_circulos['Name']==nome], how="inner", predicate="within")
        if not m_data.empty:
            final_series_int[nome] = m_data.groupby('date')['dV_final'].mean()

    # --- GRÁFICOS SÉRIE INTEGRADA ---
    print("4/4 - Gerando gráficos série integrada...")
    inst_comuns_int = [n for n in final_series_int.keys() if n in df_geo_raw['instrumento'].unique()]
    stats_int = []

    # Escala uniforme
    all_vals_plot = []
    for nome in inst_comuns_int:
        s = final_series_int[nome]
        ref = s.iloc[0]
        all_vals_plot.extend((s - ref).values)
        s_geo_all = df_geo_raw[df_geo_raw['instrumento']==nome]
        if not s_geo_all.empty:
            ref_g = s_geo_all['valor_corrigido'].iloc[0]
            all_vals_plot.extend((s_geo_all['valor_corrigido'] - ref_g).values)
    y_min = min(all_vals_plot) - 2 if all_vals_plot else -10
    y_max = max(all_vals_plot) + 2 if all_vals_plot else 10

    if inst_comuns_int:
        fig, axes = plt.subplots(len(inst_comuns_int), 1, figsize=(14, 4.5*len(inst_comuns_int)))
        if len(inst_comuns_int) == 1: axes = [axes]

        for i, nome in enumerate(inst_comuns_int):
            ax = axes[i]
            s_insar  = final_series_int[nome].sort_index()
            ref_insar = s_insar.iloc[0]
            insar_sync = s_insar - ref_insar

            # Tendência sobre Trend STL (série integrada)
            try:
                stl_int = seasonal_decompose(s_insar.interpolate().ffill().bfill(),
                                             period=12, model='additive', extrapolate_trend='freq')
                trend_int = stl_int.trend.dropna()
                trend_int_zero = trend_int - trend_int.iloc[0]
                days_trend_int = (trend_int_zero.index - trend_int_zero.index[0]).days.values
                ts_stl_int = scipy_stats.theilslopes(trend_int_zero.values, days_trend_int)
                vel_stl_int    = ts_stl_int.slope * 365.25
                vel_stl_int_lo = ts_stl_int.low_slope  * 365.25
                vel_stl_int_hi = ts_stl_int.high_slope * 365.25
                has_stl = True
            except Exception:
                has_stl = False

            # Geodesia
            s_geo_full = df_geo_raw[df_geo_raw['instrumento']==nome].sort_values('data')
            s_geo_2018 = s_geo_full[s_geo_full['data'] >= s_insar.index[0]]
            vel_g = np.nan
            if len(s_geo_2018) > 1:
                ref_g   = s_geo_2018['valor_corrigido'].iloc[0]
                vals_g  = (s_geo_2018['valor_corrigido'] - ref_g).values
                days_g  = (s_geo_2018['data'] - s_geo_2018['data'].iloc[0]).dt.days.values
                ts_g    = scipy_stats.theilslopes(vals_g, days_g)
                vel_g   = ts_g.slope * 365.25

                insar_days = (insar_sync.index - insar_sync.index[0]).days.values
                insar_at_g = np.interp(days_g, insar_days, insar_sync.values)
                rmse_int   = np.sqrt(mean_squared_error(vals_g, insar_at_g))
                r_int, p_int = scipy_stats.pearsonr(vals_g, insar_at_g)

                ax.scatter(s_geo_full['data'][s_geo_full['data'].dt.year < s_insar.index[0].year],
                           (s_geo_full['valor_corrigido'][s_geo_full['data'].dt.year < s_insar.index[0].year] - ref_g),
                           color='gray', marker='D', s=20, alpha=0.4, label='Histórico Geodesia')
                ax.scatter(s_geo_2018['data'], vals_g, color='red', marker='D', s=35,
                           label='Geodesia (2018-2023)', zorder=5)
                ax.plot(s_geo_2018['data'], ts_g.intercept + ts_g.slope * days_g,
                        color='darkred', ls=':', lw=2, label=f'Trend Geo {vel_g:.2f} mm/a')
            else:
                rmse_int, r_int, p_int = np.nan, np.nan, np.nan

            # InSAR bruto
            ax.plot(insar_sync.index, insar_sync.values, color='tab:blue', lw=1, alpha=0.4,
                    label='InSAR Integrado (2018-2023)')

            # Trend STL + IC
            if has_stl:
                ax.plot(trend_int_zero.index, trend_int_zero.values,
                        color='navy', lw=1.8, label='InSAR Trend (STL)')
                trend_fit_int = ts_stl_int.intercept + ts_stl_int.slope * days_trend_int
                ax.plot(trend_int_zero.index, trend_fit_int, color='darkblue', lw=2.5,
                        label=f'Trend InSAR(STL) {vel_stl_int:.2f} [{vel_stl_int_lo:.2f},{vel_stl_int_hi:.2f}] mm/a')
                ax.fill_between(trend_int_zero.index,
                                ts_stl_int.low_slope  * days_trend_int + ts_stl_int.intercept,
                                ts_stl_int.high_slope * days_trend_int + ts_stl_int.intercept,
                                color='darkblue', alpha=0.12)

            ax.axhline(0, color='black', lw=1)
            ax.axvline(s_insar.index[0], color='green', lw=1.5, ls='--',
                       label=f'Início InSAR ({s_insar.index[0].date()})')
            ax.set_ylim(y_min, y_max)
            ax.set_title(
                f"Ponto {nome} | RMSE={rmse_int:.2f} mm | R²={r_int**2:.3f} | p={p_int:.4f}" if not np.isnan(rmse_int)
                else f"Ponto {nome}: Comparação de Tendências (2018-2023)",
                fontweight='bold'
            )
            ax.set_ylabel("Variação (mm)")
            ax.grid(True, alpha=0.3)
            ax.legend(loc='upper left', fontsize=7.5, ncol=2)
            ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

            stats_int.append({
                'Ponto':                    nome,
                'Vel_InSAR_STL (mm/a)':     round(vel_stl_int, 3) if has_stl else np.nan,
                'IC95_low':                 round(vel_stl_int_lo, 3) if has_stl else np.nan,
                'IC95_high':                round(vel_stl_int_hi, 3) if has_stl else np.nan,
                'Vel_Geo (mm/a)':           round(vel_g, 3) if not np.isnan(vel_g) else np.nan,
                'RMSE (mm)':                round(rmse_int, 3) if not np.isnan(rmse_int) else np.nan,
                'R²_cruzado':               round(r_int**2, 3) if not np.isnan(r_int) else np.nan,
                'p_cruzado':                round(p_int, 4) if not np.isnan(p_int) else np.nan,
            })

        plt.tight_layout(); plt.show()

    print("\n--- TABELA RESUMO FINAL (SÉRIE INTEGRADA 2018-2023) ---")
    df_resumo_int = pd.DataFrame(stats_int)
    print(df_resumo_int.to_string(index=False))
